# 01 - Understand the CAMUS Dataset

Questions I should be able to answer after this notebook:

1. Where is the CAMUS dataset located?
2. How many patients are there in total?
3. How are the training / validation / testing splits defined?
4. What files are inside one patient folder?
5. What do `2CH`, `4CH`, `ED`, `ES`, and `gt` mean?
6. What is the data-type difference between the original image and the mask?
7. Why is it a segmentation task?

## 1. Big Picture

CAMUS is a cardiac ultrasound image segmentation dataset. In this project, the model needs to learn the following mapping:

```text
Input: a 2D cardiac ultrasound image
Output: a segmentation mask with the same spatial size
```

This means the model does not classify the whole image into one category. Instead, it predicts **which anatomical class each pixel belongs to**.

This type of task is called:

```text
semantic segmentation
```

In later U-Net / nnU-Net training:

```text
image = input problem
gt mask = ground-truth answer
```

## 2. Import Basic Tools

Here we only use basic Python tools to inspect the files. No model training is involved.

In [2]:
from pathlib import Path
from collections import Counter
import gzip
import struct

import numpy as np

## 3. Locate the Dataset

Inside this project folder, the CAMUS data is mainly stored in two locations:

```text
Resources/database_nifti/   # images and masks
Resources/database_split/   # official train / validation / test split
```

In [3]:
def find_project_root():
    """Find the CV_project folder that contains the CAMUS Resources directory."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.append(Path("/Users/felix/Downloads/CV_project"))

    for candidate in candidates:
        data_root = candidate / "Resources" / "database_nifti"
        split_root = candidate / "Resources" / "database_split"
        if data_root.exists() and split_root.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find CAMUS Resources. Please check that CV_project/Resources exists."
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "Resources" / "database_nifti"
SPLIT_ROOT = PROJECT_ROOT / "Resources" / "database_split"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists(), DATA_ROOT)
print("SPLIT_ROOT exists:", SPLIT_ROOT.exists(), SPLIT_ROOT)

PROJECT_ROOT: /Users/felix/Downloads/CV_project
DATA_ROOT exists: True /Users/felix/Downloads/CV_project/Resources/database_nifti
SPLIT_ROOT exists: True /Users/felix/Downloads/CV_project/Resources/database_split


## 4. Count Patients

Each `patientXXXX` folder represents one patient. First, count how many patients are available.

In [4]:
patient_dirs = sorted(DATA_ROOT.glob("patient*"))

print("Number of patients:", len(patient_dirs))
print("First 5 patients:", [p.name for p in patient_dirs[:5]])
print("Last 5 patients:", [p.name for p in patient_dirs[-5:]])

Number of patients: 500
First 5 patients: ['patient0001', 'patient0002', 'patient0003', 'patient0004', 'patient0005']
Last 5 patients: ['patient0496', 'patient0497', 'patient0498', 'patient0499', 'patient0500']


You should see 500 patients in total.

This shows that this is not a toy dataset. It is the full local CAMUS dataset.

## 5. Check Train / Validation / Test Split

In a medical image project, all data should not be mixed together for training.

The dataset is usually split into:

```text
training set   = used to train the model
validation set = used for tuning and model selection
testing set    = used for final generalization evaluation
```

In [5]:
def read_split(filename):
    path = SPLIT_ROOT / filename
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

train_patients = read_split("subgroup_training.txt")
val_patients = read_split("subgroup_validation.txt")
test_patients = read_split("subgroup_testing.txt")

print("training:", len(train_patients))
print("validation:", len(val_patients))
print("testing:", len(test_patients))
print("total:", len(train_patients) + len(val_patients) + len(test_patients))

print("First 5 training patients:", train_patients[:5])

training: 400
validation: 50
testing: 50
total: 500
First 5 training patients: ['patient0001', 'patient0002', 'patient0003', 'patient0004', 'patient0005']


For this local CAMUS dataset, the split is:

```text
training   = 400 patients
validation = 50 patients
testing    = 50 patients
```

## 6. Inspect One Patient Folder

Do not start by looking at all data at once. First inspect `patient0001`. This step is important.

The goal is to understand what one patient folder contains.

In [6]:
patient_id = "patient0001"
patient_dir = DATA_ROOT / patient_id

files = sorted(patient_dir.iterdir())
for f in files:
    print(f.name)

.DS_Store
Info_2CH.cfg
Info_4CH.cfg
MANDATORY_CITATION.md
patient0001_2CH_ED.nii.gz
patient0001_2CH_ED_gt.nii
patient0001_2CH_ED_gt.nii.gz
patient0001_2CH_ES.nii.gz
patient0001_2CH_ES_gt.nii.gz
patient0001_2CH_half_sequence.nii.gz
patient0001_2CH_half_sequence_gt.nii.gz
patient0001_4CH_ED.nii.gz
patient0001_4CH_ED_gt.nii.gz
patient0001_4CH_ES.nii.gz
patient0001_4CH_ES_gt.nii.gz
patient0001_4CH_half_sequence.nii.gz
patient0001_4CH_half_sequence_gt.nii.gz


### File Name Explanation

Take this file as an example:

```text
patient0001_2CH_ED.nii.gz
```

The file name can be decomposed as:

```text
patient0001 = patient ID
2CH         = two-chamber view
ED          = end-diastole
nii.gz      = NIfTI medical image format
```

If the file name contains `_gt`:

```text
patient0001_2CH_ED_gt.nii.gz
```

It means ground truth, which is the manually annotated segmentation mask.

## 7. Read Info Files

`Info_2CH.cfg` and `Info_4CH.cfg` store metadata for this patient.

Important fields:

```text
ED = the frame index of end-diastole
ES = the frame index of end-systole
NbFrame = the number of frames in this view
EF = ejection fraction
ImageQuality = image quality
```

In [7]:
def read_cfg(path):
    info = {}
    for line in path.read_text().splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            info[key.strip()] = value.strip()
    return info

info_2ch = read_cfg(patient_dir / "Info_2CH.cfg")
info_4ch = read_cfg(patient_dir / "Info_4CH.cfg")

print("Info_2CH:")
for k, v in info_2ch.items():
    print(f"  {k}: {v}")

print("\nInfo_4CH:")
for k, v in info_4ch.items():
    print(f"  {k}: {v}")

Info_2CH:
  ED: 1
  ES: 18
  NbFrame: 18
  Sex: F
  Age: 56
  ImageQuality: Good
  EF: 54
  FrameRate: 48.4

Info_4CH:
  ED: 1
  ES: 20
  NbFrame: 20
  Sex: F
  Age: 56
  ImageQuality: Good
  EF: 54
  FrameRate: 48.4


For `patient0001`, you should see:

```text
2CH: ED = 1, ES = 18
4CH: ED = 1, ES = 20
EF = 54
ImageQuality = Good
```

This shows that the ES frame index can be different between views for the same patient.

## 8. Inspect NIfTI Header

`.nii.gz` is not a normal JPG/PNG image. It is a medical image array file.

Here we only read the NIfTI header to inspect image size, spacing, and data type.

In [8]:
def inspect_nii_header(path):
    raw = gzip.open(path, "rb").read(348)
    endian = "<" if struct.unpack("<i", raw[:4])[0] == 348 else ">"
    dim = struct.unpack(endian + "8h", raw[40:56])
    pixdim = struct.unpack(endian + "8f", raw[76:108])
    datatype = struct.unpack(endian + "h", raw[70:72])[0]
    bitpix = struct.unpack(endian + "h", raw[72:74])[0]
    vox_offset = struct.unpack(endian + "f", raw[108:112])[0]
    return {
        "dim": dim,
        "spacing": pixdim,
        "datatype": datatype,
        "bitpix": bitpix,
        "vox_offset": vox_offset,
    }

sample_image = patient_dir / "patient0001_2CH_ED.nii.gz"
sample_mask = patient_dir / "patient0001_2CH_ED_gt.nii.gz"

for path in [sample_image, sample_mask]:
    header = inspect_nii_header(path)
    print(path.name)
    print("  dim:", header["dim"][:4])
    print("  spacing:", tuple(round(x, 4) for x in header["spacing"][1:4]))
    print("  datatype:", header["datatype"])
    print("  bitpix:", header["bitpix"])
    print()

patient0001_2CH_ED.nii.gz
  dim: (2, 549, 389, 1)
  spacing: (0.308, 0.308, 1.0)
  datatype: 16
  bitpix: 32

patient0001_2CH_ED_gt.nii.gz
  dim: (2, 549, 389, 1)
  spacing: (0.308, 0.308, 1.0)
  datatype: 16
  bitpix: 32



You should see something like:

```text
dim = (2, 549, 389, 1)
spacing = (0.308, 0.308, 1.0)
datatype = 16
bitpix = 32
```

Here, `549 x 389` represents the image width and height. `spacing` represents the physical size of each pixel.

## 9. Load Image and Mask Arrays

Here we actually load `.nii.gz` files as numpy arrays.

Note: this is only for understanding the data format. In a formal project, `nibabel` or `SimpleITK` is usually used.

In [9]:
def load_nii_array(path):
    raw = gzip.open(path, "rb").read()
    endian = "<" if struct.unpack("<i", raw[:4])[0] == 348 else ">"
    dim = struct.unpack(endian + "8h", raw[40:56])
    datatype = struct.unpack(endian + "h", raw[70:72])[0]
    vox_offset = int(struct.unpack(endian + "f", raw[108:112])[0])

    dtype_map = {
        2: np.uint8,
        4: np.int16,
        8: np.int32,
        16: np.float32,
        64: np.float64,
        256: np.int8,
        512: np.uint16,
        768: np.uint32,
    }

    shape = tuple(reversed(dim[1:1 + dim[0]]))
    arr = np.frombuffer(raw[vox_offset:], dtype=dtype_map[datatype], count=int(np.prod(shape))).reshape(shape)
    return np.squeeze(arr)

image = load_nii_array(sample_image)
mask = load_nii_array(sample_mask).astype(np.int64)

print("image shape:", image.shape)
print("image dtype:", image.dtype)
print("image min/max:", float(image.min()), float(image.max()))

print("\nmask shape:", mask.shape)
print("mask dtype:", mask.dtype)
print("mask labels:", np.unique(mask).tolist())

image shape: (389, 549)
image dtype: float32
image min/max: 0.0 255.0

mask shape: (389, 549)
mask dtype: int64
mask labels: [0, 1, 2, 3]


Key idea:

```text
numbers in image = grayscale intensity values, for example 0 to 255
numbers in mask  = class labels, for example 0, 1, 2, 3
```

Therefore:

```text
image = the input image given to the model
mask  = the target answer that the model needs to learn to predict
```

## 10. Understand One Training Sample

For U-Net, one training sample can be written as:

```text
X = patient0001_2CH_ED.nii.gz
y = patient0001_2CH_ED_gt.nii.gz
```

where:

```text
X.shape = H x W
y.shape = H x W
```

During deep learning training, they are reshaped into:

```text
X.shape = batch x channel x height x width
y.shape = batch x height x width
```

Because this is a grayscale image, `channel = 1`.

In [10]:
image_norm = image.astype(np.float32) / 255.0

x_single = image_norm[None, :, :]      # channel, height, width
y_single = mask                        # height, width

x_batch = x_single[None, :, :, :]      # batch, channel, height, width
y_batch = y_single[None, :, :]         # batch, height, width

print("single image X:", x_single.shape)
print("single mask y:", y_single.shape)
print("batch X:", x_batch.shape)
print("batch y:", y_batch.shape)

single image X: (1, 389, 549)
single mask y: (389, 549)
batch X: (1, 1, 389, 549)
batch y: (1, 389, 549)


If there are 4 classes, the U-Net output is usually:

```text
prediction.shape = batch x class x height x width
```

For this CAMUS example:

```text
prediction.shape = 1 x 4 x H x W
```

Then we take the maximum along the class dimension to obtain the final predicted mask:

```text
predicted_mask.shape = 1 x H x W
```

## 11. Dataset-Level Metadata Summary

Now compute a simple metadata summary over the 500 patients: age, sex, image quality, and EF distribution.

This is useful for reporting because it shows that you did not only inspect one file, but also understood the overall dataset structure.

In [11]:
ages = []
efs = []
sex_counter = Counter()
quality_counter = Counter()

for p in patient_dirs:
    info = read_cfg(p / "Info_2CH.cfg")
    if "Age" in info:
        ages.append(int(info["Age"]))
    if "EF" in info:
        efs.append(int(info["EF"]))
    if "Sex" in info:
        sex_counter[info["Sex"]] += 1
    if "ImageQuality" in info:
        quality_counter[info["ImageQuality"]] += 1

print("Age min/mean/max:", min(ages), round(float(np.mean(ages)), 1), max(ages))
print("EF min/mean/max:", min(efs), round(float(np.mean(efs)), 1), max(efs))
print("Sex:", dict(sex_counter))
print("ImageQuality:", dict(quality_counter))

Age min/mean/max: 18 65.1 93
EF min/mean/max: 5 44.4 81
Sex: {'F': 170, 'M': 330}
ImageQuality: {'Good': 217, 'Medium': 214, 'Poor': 69}


## 12. What I Learned

Summary in my own words:

1. The local CAMUS dataset contains 500 patients.
2. The official split is 400 training, 50 validation, and 50 testing patients.
3. Each patient has two views: 2CH and 4CH.
4. Each view has two key cardiac phases: ED and ES.
5. `_gt.nii.gz` is the ground-truth segmentation mask.
6. The image stores grayscale intensities, while the mask stores class labels.
7. U-Net performs semantic segmentation: it predicts a class for each pixel.

## 13. Questions I Still Have

Write open questions here and answer them later:

1. Which cardiac structures do labels 1, 2, and 3 correspond to?
2. Why does EF calculation mainly require the LV mask?
3. Why can U-Net output a mask with the same spatial size as the input image?
4. How is Dice score calculated?
5. What is the difference between nnU-Net and a standard U-Net?

## 14. Short Explanation for Meeting

Possible explanation for a meeting:

> I started by understanding the CAMUS dataset itself. The local dataset contains 500 patients with 2D cardiac ultrasound images. The official split contains 400 training, 50 validation, and 50 testing patients. Each patient has two views, 2CH and 4CH, and provides images at two key cardiac phases, ED and ES, together with manually annotated segmentation masks. The original image stores grayscale ultrasound intensity, while the ground-truth mask stores multi-class pixel labels with the same spatial size. Therefore, this project is a semantic segmentation task: the model needs to predict the cardiac structure class for each pixel.